[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aprendizaje-automatico-dc-uba-ar/material/blob/main/notebooks/lferrer/2_multiclass_ecs-published.ipynb)

# EC examples for a general multi-class or binary classification problem

Comparison of EC values for different cost matrices, including two with abstention decisions with different costs.

In [1]:
# ! git clone https://github.com/luferrer/expected_cost.git

In [2]:
# absolute paths    
PATH_TO_EXPECTED_COST = "..."

In [4]:
import sys
sys.path.append(PATH_TO_EXPECTED_COST)

In [5]:
import numpy as np
from expected_cost import ec, utils
from expected_cost.data import get_llks_for_multi_classif_task

In [6]:
# Parameters of the simulation

N = 10000        # total number of samples
std = 0.20       # within-class standard deviation of the features, determines the difficulty of the problem
P0 = 0.8         # Prior for class 0
num_targets = 10 # number of classes

priors = np.array([P0] + [(1 - P0) / (num_targets - 1)] * (num_targets - 1))

# Generate the log-likelihoods and log posteriors

targets, _, llks = get_llks_for_multi_classif_task('gaussian_sim', priors=priors, sim_params={'feat_std': std}, N=N)
logpost = utils.llks_to_logpost(llks, priors)

# Define various costs matrices

costs = {}

# Standard 0-1 cost
costs['EC1'] = ec.CostMatrix.zero_one_costs(num_targets)

# Balanced error rate. The costs are inversely proportional to the priors.
costm = (1 - np.eye(num_targets))/np.atleast_2d(priors).T/num_targets
costs['ECb']  = ec.CostMatrix(costm)

# A 0-1 cost with the last rwo replaced by 100 to simulate a case in which the errors in that class
# are much more costly than the errors in other classes.
costm = 1 - np.eye(num_targets)
costm[-1,:] *= 100
costs['EC5']  = ec.CostMatrix(costm)

# Finally, two cost functions with abstention options with different cost.
costs['ECabs1'] = ec.CostMatrix.zero_one_costs(num_targets, abstention_cost=0.05)
costs['ECabs2'] = ec.CostMatrix.zero_one_costs(num_targets, abstention_cost=0.2)


In [7]:
# Print the best naive decision for each cost matrix. When all decisions are equally costly, it
# just picks one of them. This happens for ECb.

for costn, cost in costs.items():
    naive_dec = np.argmin(np.dot(priors.T, cost.get_matrix()))
    print(f"Best naive decision for {costn:7s}: {naive_dec}")
print("")

# Table Header

print("Raw and normalized EC values.")
print("For the ECs with an abstention option, the percentage of samples where that decision was made (%Abs) is included.")
print("")

print(f"Decisions ", end='')
for costn, cost in costs.items():
    print(f"| {costn:10s} ", end='')
    if 'abs' in costn:
        print(f"     ", end='')
print("")
print(f"          ", end='')
for costn, cost in costs.items():
    print(f"| Raw   Norm ", end='')
    if 'abs' in costn:
        print(f"%Abs ", end='')
print("")

# Argmax decisions, which are the same for all cost matrices

argmax_decisions = np.argmax(logpost, axis=-1)

# Print the various ECs for each of the decision algorithms

for dec in ['Naive', 'Argmax', 'Bayes']:

    print(f"{dec:6s}   ", end='')

    for costn, cost in costs.items():

        if dec == 'Argmax':
            decisions = argmax_decisions
        elif dec == 'Bayes':
            decisions, _ = ec.bayes_decisions(logpost, cost, score_type='log_posteriors')
        elif dec == 'Naive':
            decisions = np.ones_like(targets) * np.argmin(np.dot(priors.T, cost.get_matrix()))
        else:
            print("Unknown decision algorithm")
            continue

        ecval  = ec.average_cost(targets, decisions, cost, adjusted=False)
        ecvaln = ec.average_cost(targets, decisions, cost, adjusted=True)

        print(f" | {ecval:5.2f}{ecvaln:5.2f}", end='')
        norm = np.min(np.dot(priors.T, cost.get_matrix()))
        if 'abs' in costn:
            perc_abs = np.sum(decisions == num_targets) / len(decisions) * 100
            print(f"{perc_abs:5.0f}", end='')

    print("")



Best naive decision for EC1    : 0
Best naive decision for ECb    : 0
Best naive decision for EC5    : 9
Best naive decision for ECabs1 : 10
Best naive decision for ECabs2 : 0

Raw and normalized EC values.
For the ECs with an abstention option, the percentage of samples where that decision was made (%Abs) is included.

Decisions | EC1        | ECb        | EC5        | ECabs1          | ECabs2          
          | Raw   Norm | Raw   Norm | Raw   Norm | Raw   Norm %Abs | Raw   Norm %Abs 
Naive     |  0.20 1.00 |  0.90 1.00 |  0.98 1.00 |  0.05 1.00  100 |  0.20 1.00    0
Argmax    |  0.06 0.32 |  0.28 0.31 |  0.36 0.37 |  0.06 1.29    0 |  0.06 0.32    0
Bayes     |  0.06 0.32 |  0.23 0.26 |  0.08 0.08 |  0.02 0.35   25 |  0.05 0.23   12
